In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns 

In [2]:
df = pd.read_csv("../data/train.csv")
df_t = pd.read_csv("../data/test.csv")

In [4]:
test_ids = df_t["id"].copy()

df.drop(["id"], axis=1, inplace=True)
df_t.drop(["id"], axis=1, inplace=True)

In [7]:
df["Will_Buy_EV_num"] = df["Will_Buy_EV"].map({
    "No": 0,
    "Yes": 1
})

In [8]:
def create_features(data):

    data = data.copy()

    data["Subsidy_flag"] = data["Subsidy_Available"].map({
        "No": 0,
        "Yes": 1
    })

    data["Home_Charging_flag"] = data["Home_Charging_Possible"].map({
        "No": 0,
        "Yes": 1
    })

    data["total_Charging_Availability"] = (
        data["Charging_Stations_Near_Home"]
        + data["Charging_Stations_Near_Work"]
    )

    data["Income_x_Subsidy"] = (
        data["Annual_Income_USD"]
        * data["Subsidy_flag"]
    )

    data["EnvironmentalConcern_x_Subsidy"] = (
        data["Environmental_Concern_Level"]
        * data["Subsidy_flag"]
    )

    data["No_Home_Charging"] = (
        1 - data["Home_Charging_flag"]
    )

    data["NoHomeCharging_x_NearHomeStations"] = (
        data["No_Home_Charging"]
        * data["Charging_Stations_Near_Home"]
    )

    data["City_HomeCharging"] = (
        data["City_Type"].astype(str)
        + "_"
        + data["Home_Charging_Possible"].astype(str)
    )

    data["Charging_Per_Commute"] = (
        data["total_Charging_Availability"]
        / (data["Daily_Commute_km"] + 1)
    )

    data["Commute_x_TotalCharging"] = (
        data["Daily_Commute_km"]
        * data["total_Charging_Availability"]
    )

    return data


df = create_features(df)
df_t = create_features(df_t)

In [9]:
new_features = [
    "Subsidy_flag",
    "Home_Charging_flag",
    "total_Charging_Availability",
    "Income_x_Subsidy",
    "EnvironmentalConcern_x_Subsidy",
    "No_Home_Charging",
    "NoHomeCharging_x_NearHomeStations",
    "City_HomeCharging",
    "Charging_Per_Commute",
    "Commute_x_TotalCharging"
]

In [10]:
df.drop(
    columns=[
        "Will_Buy_EV_num",
        "Home_Charging_Possible",
        "Subsidy_Available",
        "No_Home_Charging",
        "Gender"
    ],
    inplace=True
)

df_t.drop(
    columns=[
        "Home_Charging_Possible",
        "Subsidy_Available",
        "No_Home_Charging",
        "Gender"
    ],
    inplace=True
)

In [11]:
df.duplicated().sum()

np.int64(100)

In [12]:
df.drop_duplicates(inplace=True)

In [15]:
df.head()

,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,City_Type,Current_Car_Type,Range_Anxiety_Level,Will_Buy_EV,Subsidy_flag,Home_Charging_flag,total_Charging_Availability,Income_x_Subsidy,EnvironmentalConcern_x_Subsidy,NoHomeCharging_x_NearHomeStations,City_HomeCharging,Charging_Per_Commute,Commute_x_TotalCharging
0,66,92887.0,23.4,2,3,7,1.0,Suburban,Sedan,Low,No,0,1,10,0.0,0.0,0,Suburban_Yes,0.409836,234.0
1,38,30000.0,5.0,1,2,2,4.0,Rural,SUV,Low,No,0,1,4,0.0,0.0,0,Rural_Yes,0.666667,20.0
2,26,94389.0,36.8,1,8,15,5.0,Urban,Sedan,Low,Yes,1,0,23,94389.0,5.0,8,Urban_No,0.608466,846.4
3,66,73580.0,23.7,2,6,9,3.0,Suburban,Hatchback,Low,No,0,1,15,0.0,0.0,0,Suburban_Yes,0.607287,355.5
4,54,57898.0,50.8,1,2,3,3.0,Suburban,Hatchback,Low,No,0,1,5,0.0,0.0,0,Suburban_Yes,0.096525,254.0


In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 668565 entries, 0 to 668664
Data columns (total 20 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   Age                                668565 non-null  int64  
 1   Annual_Income_USD                  668565 non-null  float64
 2   Daily_Commute_km                   668565 non-null  float64
 3   Number_of_Cars_Owned               668565 non-null  int64  
 4   Charging_Stations_Near_Home        668565 non-null  int64  
 5   Charging_Stations_Near_Work        668565 non-null  int64  
 6   Environmental_Concern_Level        668565 non-null  float64
 7   City_Type                          668565 non-null  object 
 8   Current_Car_Type                   668565 non-null  object 
 9   Range_Anxiety_Level                668565 non-null  object 
 10  Will_Buy_EV                        668565 non-null  object 
 11  Subsidy_flag                       668565 no

In [13]:
from sklearn.model_selection import train_test_split 
from imblearn.pipeline import Pipeline   
from sklearn.compose import ColumnTransformer 
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
    LabelEncoder,
    FunctionTransformer
)
from xgboost import XGBClassifier 
from sklearn.metrics import roc_auc_score, roc_curve 
from imblearn.over_sampling import SMOTE

In [14]:
Num_cols = df.select_dtypes(exclude = 'object').columns
Cat_cols = df.select_dtypes(include='object').columns